# Financial Data Extraction

Module: Markets and Data

## Lesson summary

This notebook introduces provider-aware data extraction for market prices, macroeconomic indicators, reference data, corporate data, and metadata. The default run uses deterministic classroom panels, while live extraction is an instructor-approved extension through reusable clients in `src`.

The purpose is to teach a reproducible extraction workflow: choose sources deliberately, document metadata, preserve raw observations, use cache, and separate retrieval from cleaning, validation, and analysis {cite}`wilkinson2016fair`.

## Learning objectives

By the end of this lesson, students should be able to:

- distinguish official, commercial, and convenience data providers;
- classify market, macroeconomic, reference, corporate, and metadata sources;
- evaluate providers by authority, coverage, definition, frequency, latency, revisions, accessibility, reliability, licensing, and reproducibility;
- document provider, field, calendar, currency, and known limitations before analysis;
- separate raw, interim, processed, metadata, and cache layers;
- load classroom-safe price and macro panels without network access;
- use `MarketDataClient` as the live-data entry point instead of writing provider logic inside notebooks;
- explain why cached extracts are part of a reproducible quantitative workflow.

## Setup

In [ ]:
import os

import pandas as pd

from src.banxico import example_banxico_catalog
from src.fred import fred_series_catalog
from src.market_data import MarketDataClient, dashboard_data_inventory, synthetic_macro_panel, synthetic_price_panel
from src.market_data_quality import data_quality_report, source_inventory_template

DATA_MODE = os.getenv("DATA_MODE", "offline")

## Practical questions

This part of the module moves from market concepts to data work. It answers five practical questions:

1. Where do financial and macroeconomic data come from?
2. How should data be downloaded and stored?
3. How can raw data be cleaned and validated?
4. How are prices transformed into returns?
5. How can a reproducible data pipeline be built for the Mexican market?

## Source taxonomy

A financial dataset is only useful if the analyst understands its source, definition, frequency, coverage, limitations, update process, and usage restrictions. Data should not be treated as neutral input; every source reflects institutional rules, market conventions, vendor decisions, and sometimes revision policies.

| Data type | Examples | Typical source family | Main caution |
| --- | --- | --- | --- |
| Market data | prices, volumes, bid-ask quotes, index levels, yields, spreads, exchange rates | exchanges, vendors, brokers, public portals | fields and calendars differ by instrument and provider |
| Macroeconomic data | inflation, rates, GDP, employment, industrial activity, trade, monetary aggregates | central banks, statistical offices, FRED, ALFRED | publication lags and revisions matter |
| Reference data | tickers, identifiers, issuers, calendars, sectors, maturity dates, coupons | exchanges, vendors, internal dictionaries | stale mappings can corrupt joins |
| Corporate data | statements, dividends, splits, earnings, debt issuance, corporate actions | issuers, exchanges, vendors | events affect return calculation |
| Metadata | units, frequency, methodology, retrieval date, license, adjustment policy | source docs and local data dictionaries | missing metadata makes results hard to audit |

Market data are often transaction-driven and high-frequency. Macroeconomic data are usually publication-driven, lower-frequency, and sometimes revised after initial release. That difference affects how series should be aligned before analysis.

## Provider inventory

Start with the source map before requesting data. A provider choice is a modeling assumption because it determines fields, calendars, missing values, revisions, and usage limits.

In [ ]:
dashboard_data_inventory()

In [ ]:
example_banxico_catalog()

In [ ]:
fred_series_catalog()

## Source selection criteria

Do not choose a source only because it is convenient. Evaluate it explicitly.

| Criterion | Practical question |
| --- | --- |
| Authority | Is the source official, licensed, audited, or widely accepted? |
| Coverage | Which instruments, markets, dates, and frequencies are included? |
| Definition | Are units, methodology, and adjustments documented? |
| Frequency | Is the data daily, weekly, monthly, quarterly, intraday, or event-based? |
| Latency | How quickly is the data updated after publication or trading? |
| Revisions | Can historical values change after publication? |
| Accessibility | Is there an API, bulk download, manual file, or web interface? |
| Reliability | Are outages, missing values, or schema changes common? |
| Licensing | Can the data be stored, redistributed, published, or commercialized? |
| Reproducibility | Can another analyst retrieve the same data with the same code? |

For educational work, use a simple hierarchy: official sources for macro and regulatory data, exchange or licensed sources for official market data, public portals and wrappers for exploratory learning, local cached datasets for reproducible exercises, and synthetic data when licenses prevent redistribution.

## Source inventory template

In [ ]:
inventory = source_inventory_template()
inventory.loc[0] = {
    "provider": "Yahoo Finance",
    "instrument_or_variable": "^MXX, AMX.MX, WALMEX.MX",
    "frequency": "daily",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "adjusted close",
    "currency": "MXN",
    "calendar": "trading days",
    "known_limitations": "unofficial endpoint and possible rate limits",
}
inventory.loc[1] = {
    "provider": "Banxico SIE",
    "instrument_or_variable": "SF61745, SF60633, SF43718",
    "frequency": "daily or publication frequency",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "published value",
    "currency": "MXN or percent",
    "calendar": "Mexican publication calendar",
    "known_limitations": "requires token; macro and rate series can have missing publication dates",
}
inventory.loc[2] = {
    "provider": "FRED",
    "instrument_or_variable": "DGS10, DEXMXUS, MEXCPALTT01IXNBM",
    "frequency": "daily or monthly",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "value",
    "currency": "USD, MXN, index, or percent",
    "calendar": "US publication calendar",
    "known_limitations": "series revisions, publication lags, and mixed frequencies",
}
inventory.loc[3] = {
    "provider": "INEGI",
    "instrument_or_variable": "inflation or economic activity indicator",
    "frequency": "monthly or quarterly",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "published value",
    "currency": "index, percent, or level",
    "calendar": "Mexican statistical publication calendar",
    "known_limitations": "publication lags, revisions, and indicator-specific units",
}
inventory.loc[4] = {
    "provider": "BMV or licensed market data source",
    "instrument_or_variable": "Mexican equities, ETFs, FIBRAs, or indices",
    "frequency": "daily or intraday",
    "start": "2021-01-01",
    "end": "2024-12-31",
    "field": "price, volume, index level, or market data product field",
    "currency": "MXN",
    "calendar": "Mexican trading calendar",
    "known_limitations": "licensing, redistribution limits, and instrument-specific field definitions",
}
inventory

## Extraction, storage, and cache

A reproducible analysis should not depend on repeatedly calling live APIs every time a notebook runs. Extraction should make it possible to answer a simple audit question: **which data were used to produce this result?**

A useful educational architecture separates the data workflow into layers:

| Layer | Purpose |
| --- | --- |
| `raw/` | Store data exactly as downloaded from the source |
| `interim/` | Store partially cleaned or standardized data |
| `processed/` | Store analysis-ready datasets |
| `external/` | Store manually provided or licensed files |
| `metadata/` | Store dictionaries, source notes, schemas, and assumptions |
| `cache/` | Store reusable API responses or intermediate outputs |

The raw layer should remain close to the original source. Cleaning and transformations should happen later so that another analyst can inspect what changed.

Cache matters because it reduces dependency on live API availability, avoids repeated downloads, improves notebook speed, reduces rate-limit risk, supports offline reproduction, and preserves the snapshot used in a specific run. Cache is not the same as final storage; it is a convenience layer around reproducible inputs.

## Extraction design

A basic extraction function should follow a predictable structure:

1. Validate input parameters.
2. Build the request.
3. Call the source.
4. Handle errors.
5. Save the raw response when possible.
6. Parse the response.
7. Standardize columns.
8. Attach metadata.
9. Save output.
10. Return a clean object to the user.

For APIs that require keys, tokens, or credentials, secrets should not be hard-coded in notebooks or committed to the repository.

## File formats and configuration

Choose formats according to the analytical purpose.

| Format | Suggested use |
| --- | --- |
| CSV | Simple tabular data and teaching examples |
| Parquet | Efficient columnar storage for larger datasets |
| JSON | API responses and metadata |
| YAML | Configuration files |
| SQLite | Small relational local databases |
| Markdown | Assumptions, documentation, and data notes |

A configuration file can keep assumptions visible:

```yaml
project:
  name: market_and_data
  timezone: America/Mexico_City

sources:
  banxico:
    frequency: daily
    cache: true

  fred:
    frequency: monthly
    cache: true

series:
  exchange_rate:
    provider: banxico
    id: example_series_id

  inflation:
    provider: inegi
    id: example_indicator_id
```

The goal is not to make the project complex. The goal is to make assumptions visible.

## Audit trail

Each extraction run should create a minimal audit trail:

| Field | Example |
| --- | --- |
| source name | Banxico SIE |
| series identifier | `SF43718` |
| retrieval date | date when the request ran |
| start and end date | requested period |
| rows | number of observations returned |
| missing values | count after parsing |
| file path | raw or cached location |
| code version | commit hash or notebook version, when available |
| warnings | provider errors, schema changes, or unexpected gaps |

This is the first line of defense before any model, dashboard, or interpretation.

## Classroom extraction

The book build should not depend on external APIs. These deterministic panels mimic the structure students will receive from live providers.

In [ ]:
prices = synthetic_price_panel(periods=260)
macro = synthetic_macro_panel(periods=60)

prices.tail()

In [ ]:
macro.tail()

In [ ]:
data_quality_report(prices)

## Live-data extension

Live extraction stays behind an explicit environment switch. Run it locally only when credentials, network access, and rate limits are approved for class.

In [ ]:
if DATA_MODE == "live":
    client = MarketDataClient()
    live_prices = client.yahoo_prices(
        ["^MXX", "AMX.MX", "WALMEX.MX"],
        start="2021-01-01",
        end="2024-12-31",
    )
    live_macro = client.fred.fetch_series_group(
        ["DGS10", "DEXMXUS", "MEXCPALTT01IXNBM"],
        start="2021-01-01",
        end="2024-12-31",
    )
else:
    live_prices = prices
    live_macro = macro

live_prices.tail()

In [ ]:
live_macro.tail()

## Extraction checklist

| Question | Why it matters |
| --- | --- |
| Is the provider official, commercial, open, or unofficial? | The answer affects reliability, legal use, and reproducibility. |
| What field was used? | Close, adjusted close, settlement, yield, and index levels imply different transformations. |
| What calendar does the source follow? | Calendar mismatches change missingness, correlations, and volatility. |
| Are credentials or rate limits involved? | Hidden credentials and unstable limits make notebooks hard to reproduce. |
| Was the raw extract cached? | Cached data gives students a stable audit trail before modeling. |